<a href="https://colab.research.google.com/github/Dwayne-tech/DML/blob/main/Deep_Learning_and_Neural_Networks_Checkpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# Installing the required packages
!pip install streamlit
!pip install pyngrok
!pip install streamlit-option-menu
!pip install SpeechRecognition
!pip install pyaudio
!pip install nltk
!pip install pyaudio
!pip install PyAudio‑0.2.11‑cp310‑cp310‑manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install -U nltk && python -m nltk.downloader punkt wordnet stopwords gutenberg inaugural punkt_tab

  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pyaudio
Failed to build pyaudio
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pyaudio)
  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely

In [21]:
%%writefile app.py
import nltk
nltk.download('punkt_tab', download_dir='/usr/share/nltk_data/')
import streamlit as st
import speech_recognition as sr
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, gutenberg, inaugural
from nltk.tokenize import sent_tokenize
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download required NLTK resources
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('gutenberg')
nltk.download('inaugural')
nltk.download('punkt_tab')

# Create default data.txt if not found
try:
    with open('data.txt', 'r') as file:
        corpus = file.read().splitlines()
except FileNotFoundError:
    st.warning("data.txt not found - using sample conversation data")
    try:
        hamlet_text = gutenberg.raw('shakespeare-hamlet.txt')
        inaugural_text = inaugural.raw('1789-Washington.txt')

        corpus = [
            "Hello! How can I help you?",
            "What's your name?",
            "Good morning!",
            "Good evening!",
            "How are you?",
            "I'm a chatbot designed to help answer questions.",
            "Could you please rephrase that?",
            "That's interesting, tell me more.",
            "Thank you!",
            "Goodbye!"
        ]

        try:
            corpus += sent_tokenize(hamlet_text)[:50]
            corpus += sent_tokenize(inaugural_text)[:30]
        except LookupError:
            st.error("NLTK tokenizer missing - using simple split instead")
            corpus += hamlet_text.split('. ')[:50]
            corpus += inaugural_text.split('. ')[:30]

    except Exception as e:
        st.error(f"Error loading sample data: {str(e)}")
        corpus = ["Hello!", "Goodbye!"]

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
punctuations = string.punctuation

def preprocess(text):
    tokens = nltk.word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words and token not in punctuations]
    return ' '.join(tokens)

processed_corpus = [preprocess(sentence) for sentence in corpus]
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_corpus)

def get_response(user_input):
    processed_input = preprocess(user_input)
    input_vec = tfidf_vectorizer.transform([processed_input])
    similarities = cosine_similarity(input_vec, tfidf_matrix)
    max_index = similarities.argmax()
    return corpus[max_index]

def transcribe_speech():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        st.info("Speak now...")
        try:
            audio = recognizer.listen(source, timeout=5)
            text = recognizer.recognize_google(audio)
            return text
        except sr.WaitTimeoutError:
            st.error("Listening timed out. Please try again.")
            return None
        except sr.UnknownValueError:
            st.error("Could not understand audio. Please try again.")
            return None
        except sr.RequestError as e:
            st.error(f"Could not request results; {e}")
            return None

# Streamlit app
st.title("Speech-Enabled Chatbot")

# Add unique key to radio widget
input_type = st.radio(
    "Choose input type:",
    ("Text", "Speech"),
    key="input_type_radio"  # Unique key added here
)

user_input = None

if input_type == "Text":
    user_input = st.text_input("Enter your message:", key="text_input")
else:
    if st.button("Start Recording", key="record_button"):
        user_input = transcribe_speech()

if user_input:
    st.write(f"User input: {user_input}")
    response = get_response(user_input)
    # Add unique key to text_area
    st.text_area(
        "Chatbot Response:",
        value=response,
        height=100,
        key="response_area"  # Unique key added here
    )

Overwriting app.py


In [ ]:
import subprocess
from pyngrok import ngrok
import time
import sys

# Set Ngrok authtoken
ngrok.set_auth_token("2rtFOBrMcUwbOtZblyreHGz8Ivm_Z3itCyvrSmVaYJPqif51")

def run_streamlit():
    # Start Streamlit server in the background
    streamlit_process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    return streamlit_process

if __name__ == "__main__":
    # Start Streamlit
    process = run_streamlit()

    # Wait for server to start
    time.sleep(5)

    try:
        # Create Ngrok tunnel
        public_url = ngrok.connect(addr="8501", proto="http", bind_tls=True)
        print(f"Your app is available at: {public_url.public_url}")

        # Keep the script running
        process.wait()
    except KeyboardInterrupt:
        print("Shutting down server...")
        process.terminate()
        ngrok.kill()

Your app is available at: https://2754-35-196-211-155.ngrok-free.app
